# 01-theme_mech_ai_assessment

Assess each overview against the configured mechanics and themes using its parsed text, filename and rendered pages. The result table has one current assessment per game. Its first 112 columns match the attached CSV header. Additional columns hold art style, franchise clues, provenance and a YAML copy of the assessment.

Place this notebook beside `assessment.py`. The two YAML files can go beside them or together in the `config/` subfolder. All model calls use Databricks SQL `ai_query`. The driver needs PyYAML, Delta, access to the parser's image Volume, and query access to the configured Maverick endpoint. Use one active job run per output table.

Edit the `run` section of `config.yaml` for tables, model endpoint, game limit, overwrite and optional game keys. The same file holds instructions and the CSV layout. Edit `theme.yaml` for all active theme and mechanic definitions. The combined Poker and Cards score includes keno. Pop Culture and Film Noir each have their own score.

In [0]:
# 1. Load configuration and the output contract.
from pathlib import Path
import sys
import importlib
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F

PROJECT_ROOT = Path.cwd()  # Set explicitly if the notebook runs from a different working directory.
sys.path.insert(0, str(PROJECT_ROOT))
import assessment
importlib.reload(assessment)

cfg, contract, rules, catalog, feature_columns, assessment_hash = assessment.load_config(PROJECT_ROOT)
types = assessment.column_types(feature_columns, contract["additional_columns"])
source, output, attempts = (cfg[k] for k in ("source_table", "output_table", "attempts_table"))
run_id = str(uuid.uuid4())
print(f"Run {run_id} | {len(catalog['mechanics'])} mechanics | {len(catalog['themes'])} themes")
print(f"At most {cfg['game_limit']} eligible games (-1 = all) | overwrite={cfg['overwrite']} | {output}")

In [0]:
# 2. Initialize the two Delta tables. New feature columns are added when a category is added.
def schema_sql(fields):
    return ", ".join(f"`{name}` {dtype}" for name, dtype in fields.items())

attempt_types = {
    "run_id":"STRING", "game_name_key":"STRING", "source_path":"STRING", "attempted_at":"TIMESTAMP",
    "status":"STRING", "error_message":"STRING", "assessment_hash":"STRING",
}
for table_name, fields in ((output, types), (attempts, attempt_types)):
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} ({schema_sql(fields)}) USING DELTA "
              "TBLPROPERTIES ('delta.isolationLevel' = 'Serializable')")
    existing = {f.name: f.dataType.simpleString().upper() for f in spark.table(table_name).schema}
    missing = {k:v for k,v in fields.items() if k not in existing}
    incompatible = {k:(existing[k],v) for k,v in fields.items() if k in existing and existing[k] != v}
    if incompatible:
        raise ValueError(f"Review incompatible columns in {table_name}: {incompatible}")
    if missing:
        spark.sql(f"ALTER TABLE {table_name} ADD COLUMNS ({schema_sql(missing)})")

In [0]:
# 3. Select distinct games before reading page images or invoking the endpoint.
# Keep the existing CSV convention for keys, including any 'overview' suffix.
normalized = F.regexp_replace(F.lower(F.trim(F.col("game_name"))), r"\s+", " ")
pages = (spark.table(source)
         .where(F.col("file_name").rlike(cfg["overview_filename_regex"]))
         .withColumn("game_name_key", normalized)
         .where(F.col("game_name_key").isNotNull() & (F.col("game_name_key") != "")))
if cfg["game_name_keys"]:
    requested = [" ".join(key.lower().split()) for key in cfg["game_name_keys"]]
    pages = pages.where(F.col("game_name_key").isin(requested))

docs = pages.select("game_name_key", "source_path").distinct()
existing = spark.table(output).select("game_name_key", "saved_at")
if cfg["overwrite"]:
    eligible = docs.join(existing, "game_name_key", "left")
else:
    eligible = docs.join(existing, "game_name_key", "left_anti").withColumn("saved_at", F.lit(None).cast("timestamp"))
# Multiple PDF paths for one legacy game key require an explicit choice in preferred_sources.
recent_attempts = spark.table(attempts).groupBy("game_name_key").agg(F.max("attempted_at").alias("last_attempt"))
eligible = (eligible.groupBy("game_name_key", "saved_at").agg(F.sort_array(F.collect_set("source_path")).alias("paths"))
            .join(recent_attempts, "game_name_key", "left")
            .orderBy(F.col("last_attempt").asc_nulls_first(), F.col("saved_at").asc_nulls_first(), "game_name_key"))
if cfg["game_limit"] != -1:
    eligible = eligible.limit(cfg["game_limit"])
selected = eligible.collect()
print(f"{len(selected)} distinct game keys selected")
if selected:
    display(spark.createDataFrame([(r.game_name_key, len(r.paths), r.paths[0]) for r in selected],
                                  "game_name_key string, source_count int, first_source_path string"))

In [0]:
# 4. Check and write one complete game at a time. A failed attempt leaves any earlier result available.
def save_game(record):
    incoming = spark.createDataFrame([record], schema=schema_sql(types))
    merge = (DeltaTable.forName(spark, output).alias("t").merge(
        incoming.alias("s"), "t.game_name_key = s.game_name_key"))
    if cfg["overwrite"]:
        updates = {name: f"s.`{name}`" for name in types if name != "first_assessed_at"}
        merge = merge.whenMatchedUpdate(set=updates)
    merge.whenNotMatchedInsert(values={name:f"s.`{name}`" for name in types}).execute()

def log_attempt(key, path, status, error=None):
    item = {"run_id":run_id, "game_name_key":key, "source_path":path, "attempted_at":datetime.now(timezone.utc),
            "status":status, "error_message":error, "assessment_hash":assessment_hash}
    spark.createDataFrame([item], schema=schema_sql(attempt_types)).write.mode("append").saveAsTable(attempts)

for game in selected:
    key = game.game_name_key
    path = game.paths[0] if len(game.paths) == 1 else None
    try:
        preferred = cfg["preferred_sources"].get(key)
        if preferred:
            if preferred not in game.paths:
                raise ValueError(f"Configured preferred source for {key} is missing. Available sources: {game.paths}")
            path = preferred
        if path is None:
            raise ValueError(f"Multiple overviews for {key}. Set preferred_sources in config.yaml. {game.paths}")
        game_pages = (pages.where((F.col("game_name_key") == key) & (F.col("source_path") == path))
                      .select("game_name", "file_name", "page_number", "page_text", "page_image_path", "parser_errors_json")
                      .orderBy("page_number").collect())
        facts, categories = assessment.assess_game(spark, cfg, rules, catalog, game_pages)
        record = assessment.project(facts, categories, contract["retired_theme_columns"])
        now = datetime.now(timezone.utc)
        record.update(game_name_key=key, game_name=game_pages[0].game_name, run_id=run_id, saved_at=now,
                      parse_status="valid_json", taxonomy_version=cfg["taxonomy_version"], source_path=path,
                      first_assessed_at=now, model_endpoint=cfg["model_endpoint"], assessment_hash=assessment_hash)
        if set(record) != set(types):
            raise ValueError(f"Output contract mismatch: {set(record) ^ set(types)}")
        save_game(record)
    except Exception as exc:
        message = f"{type(exc).__name__}: {exc}"[:3000]
        log_attempt(key, path, "failed", message)
        print(f"{key} | failed | {message}")
        continue
    log_attempt(key, path, "completed")
    print(f"{key} | completed")

In [0]:
# 5. Inspect the batch, including failure reasons, sources and positives per category.
batch_log = spark.table(attempts).where(F.col("run_id") == run_id)
display(batch_log.groupBy("status").count())
display(batch_log.where(F.col("status") == "failed").select("game_name_key", "error_message"))
batch = spark.table(output).where(F.col("run_id") == run_id)
display(batch.select("game_name", "number_of_reels", "win_count_type", "paylines_or_ways_count", "art_style",
                     "art_style_evidence", "persistence", "perceived_persistence", "franchise_candidate", "franchise_status"))
n = batch.count()
if n:
    counts = {}
    for kind, entries in catalog.items():
        prefix, suffix = ("mechanic", "present") if kind == "mechanics" else ("theme", "score")
        for key in entries:
            col = f"{prefix}_{key}_{suffix}"
            counts[col] = (kind, key)
    totals = batch.agg(*[F.count(F.when(F.col(c) > 0, 1)).alias(c) for c in counts]).first()
    display(spark.createDataFrame([(kind, key, totals[c], n, round(100 * totals[c] / n, 1))
                                   for c, (kind, key) in counts.items()],
                                  "category_type string, category string, positives long, games long, pct double")
            .orderBy("category_type", F.desc("positives"), "category"))

**Reading the results**

The `mechanics` and `themes` columns contain positive labels and citations. In `assessment_yaml`, every category also has its definition, rationale and assessed status, including `uncertain` and `not_evidenced`. The wide columns follow the CSV's positive/null convention. A confirmed mechanic scores 1. Theme scores measure prominence from 1 to 5. The summary theme score is the maximum supported theme score, or 0 when none is supported.

The eight retired poker, keno and table-game columns are null on newly assessed rows. Use `theme_poker_and_cards_score` for the combined feature. Set `overwrite: true` to reassess existing games under this taxonomy.

The single legacy win-count field is filled only for an unambiguous base-game count. See `win_configurations` in the YAML for selectable values, bonus values, or multiple win types. Missing images or invalid output produce a failed, retryable attempt. Ambiguous art can produce a successful result with an unknown style.

For a controlled recheck, put the exact game keys in `game_name_keys`, set `overwrite: true` and run again. With an empty filter the limit applies across all eligible overview keys, with the least recently attempted games first. A successful update keeps `first_assessed_at` and advances `saved_at`. The assessment hash identifies the configuration and extraction logic used. Adding a category automatically adds its wide columns. Previously assessed games receive that new assessment when selected with overwrite enabled. Review `franchise_status = 'candidate'` and the number of cases with unknown style or missing counts during the first 30-game pilot.

In [0]:
output_table= "development_042_silver_sandbox.demand_forecast.game_theme_mech_assessment_v1"
out=spark.table(output_table)
display(out)